# Laboratório 10 — Pipeline Definitivo

**QLoRA (4-bit) + RAG massivo + KV Cache + FlashAttention-2**

Execução prevista no Google Colab Free (GPU T4, 15GB VRAM).

## Setup

Instalação das dependências e imports base.

In [ ]:
!pip install -q -U transformers bitsandbytes accelerate datasets sentencepiece matplotlib

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
assert torch.cuda.is_available(), "GPU CUDA é obrigatória (use Colab com runtime GPU)"
print("GPU:", torch.cuda.get_device_name(0))

## Passo 1 — Ingestão eficiente (QLoRA 4-bit)

Carregamento do modelo base em 4 bits via `bitsandbytes` para reduzir o footprint inicial de VRAM.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()

In [ ]:
torch.cuda.synchronize()
vram_modelo_mb = torch.cuda.memory_allocated() / 1024**2
print(f"VRAM ocupada pelo modelo quantizado: {vram_modelo_mb:.1f} MB")

## Passo 2 — Simulando o RAG massivo

Em vez de invocar um pipeline de RAG completo, montamos o "contexto recuperado" concatenando abstracts reais do PubMed até atingir ~12.000 tokens — equivalente a 5 capítulos de manual médico recuperados pelo banco vetorial.

In [ ]:
from datasets import load_dataset

ds = load_dataset("pubmed_qa", "pqa_artificial", split="train", streaming=True)

trechos = []
total_chars = 0
ALVO_CARACTERES = 60_000

for row in ds:
    for paragrafo in row["context"]["contexts"]:
        trechos.append(paragrafo)
        total_chars += len(paragrafo)
        if total_chars >= ALVO_CARACTERES:
            break
    if total_chars >= ALVO_CARACTERES:
        break

contexto_massivo = "\n\n".join(trechos)
print(f"Trechos coletados: {len(trechos)}")
print(f"Caracteres totais: {len(contexto_massivo)}")

In [ ]:
PROMPT_SISTEMA = (
    "Você é um assistente médico. Com base estritamente nos trechos a seguir, "
    "redija um resumo clínico objetivo de 500 palavras destacando achados, "
    "metodologia e conclusões.\n\n"
)

prompt = PROMPT_SISTEMA + contexto_massivo + "\n\nResumo clínico:"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
n_tokens_prompt = inputs["input_ids"].shape[1]
print(f"Tokens no prompt: {n_tokens_prompt}")

## Passo 3 — O gargalo de geração (baseline sem KV Cache)

Geração de 100 tokens forçando `use_cache=False`. A cada token novo, todo o tensor Q, K, V do contexto é recalculado — complexidade O(n²) materializada de verdade. Instrumentamos tempo e pico de VRAM via `torch.cuda.max_memory_allocated`.

In [ ]:
import time

def bench_generate(model, inputs, n_novos_tokens=100, use_cache=True):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=n_novos_tokens,
            do_sample=False,
            use_cache=use_cache,
            pad_token_id=tokenizer.eos_token_id,
        )
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    peak_mb = torch.cuda.max_memory_allocated() / 1024**2
    return {"saida": out, "tempo_s": elapsed, "pico_vram_mb": peak_mb}

In [ ]:
model.config.use_cache = False

resultado_baseline = bench_generate(model, inputs, n_novos_tokens=100, use_cache=False)

print(f"[SEM KV Cache] Tempo:        {resultado_baseline['tempo_s']:.2f} s")
print(f"[SEM KV Cache] Pico de VRAM: {resultado_baseline['pico_vram_mb']:.1f} MB")

In [ ]:
def gerar_com_rastreio(model, inputs, n_novos_tokens=100, use_cache=True):
    """Loop de geração manual capturando VRAM e tempo a cada step."""
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    input_ids = inputs["input_ids"].clone()
    past_kv = None
    historico_vram, historico_tempo = [], []
    t_total = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_novos_tokens):
            t0 = time.perf_counter()
            if use_cache and past_kv is not None:
                feed = {"input_ids": input_ids[:, -1:], "past_key_values": past_kv}
            else:
                feed = {"input_ids": input_ids, "past_key_values": None}
            out = model(**feed, use_cache=use_cache)
            past_kv = out.past_key_values if use_cache else None
            proximo = out.logits[:, -1:, :].argmax(dim=-1)
            input_ids = torch.cat([input_ids, proximo], dim=-1)
            torch.cuda.synchronize()
            historico_vram.append(torch.cuda.memory_allocated() / 1024**2)
            historico_tempo.append(time.perf_counter() - t0)
    return {
        "saida": input_ids,
        "tempo_s": time.perf_counter() - t_total,
        "pico_vram_mb": torch.cuda.max_memory_allocated() / 1024**2,
        "historico_vram_mb": historico_vram,
        "historico_tempo_s": historico_tempo,
    }

rastreio_baseline = gerar_com_rastreio(model, inputs, n_novos_tokens=100, use_cache=False)
print(f"Pico VRAM baseline: {rastreio_baseline['pico_vram_mb']:.1f} MB")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(rastreio_baseline["historico_vram_mb"], label="Sem KV Cache", color="crimson")
ax.set_xlabel("Token gerado #")
ax.set_ylabel("VRAM alocada (MB)")
ax.set_title("Pegada de VRAM por step de geração — Baseline")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Passo 4 — A engenharia de otimização

Duas otimizações aplicadas ao mesmo modelo:

1. **KV Cache** (`use_cache=True`): reaproveita os tensores K e V já computados, transformando cada step de decoder de O(n²) em O(n).
2. **FlashAttention-2** (`attn_implementation="flash_attention_2"`): kernel hardware-aware que evita materializar a matriz `n×n` na HBM. Fallback para `sdpa` caso a wheel não esteja disponível no Colab.

In [ ]:
model.config.use_cache = True
print("KV Cache ativado:", model.config.use_cache)

In [ ]:
import gc

del model
gc.collect()
torch.cuda.empty_cache()

try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        attn_implementation="flash_attention_2",
    )
    attn_usado = "flash_attention_2"
except (ImportError, ValueError, RuntimeError) as exc:
    print(f"FlashAttention-2 indisponível ({type(exc).__name__}): {exc}")
    print("Fallback: usando attn_implementation='sdpa'.")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        attn_implementation="sdpa",
    )
    attn_usado = "sdpa"

model.eval()
model.config.use_cache = True
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
print(f"Atenção em uso: {attn_usado}")